# 动手练习：网络结构指标与随机图

来自《生物医学网络科学》（Biomedical Network Science）。 参考答案版。

## 任务 1：度、平均度与密度

以 Zachary 空手道俱乐部网络（$N=34$，$M=78$）为例：

1. 用公式 $\langle k\rangle = 2M/N$ 计算平均度 `k_mean`，用 $\rho = M/[N(N-1)/2]$ 计算密度 `rho`；
2. 找出度最大的节点 `hub` 和它的度 `k_max`；
3. 画出度分布 $P(k)$ 的柱状图。

期望看到：平均度约 4.6，密度约 0.14；度分布右偏，少数节点（33 号与 0 号）度特别大。

> **提示：** 可以用 `nx.density(G)` 核对密度。

In [ ]:
# 准备（直接运行）
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt
from collections import deque
from itertools import combinations

G = nx.Graph(nx.karate_club_graph().edges())   # 去掉边权，按无权网络计算
N, M = G.number_of_nodes(), G.number_of_edges()

In [ ]:
k_mean = 2 * M / N
rho = M / (N * (N - 1) / 2)
deg = dict(G.degree())
hub = max(deg, key=deg.get)
k_max = deg[hub]
print("N=%d, M=%d, 平均度=%.2f, 密度=%.3f（nx.density=%.3f）" % (N, M, k_mean, rho, nx.density(G)))
print("最大度 %d（节点 %d），是平均度的 %.1f 倍" % (k_max, hub, k_max / k_mean))
ks, counts = np.unique(list(deg.values()), return_counts=True)
plt.figure(figsize=(5, 3))
plt.bar(ks, counts / N)
plt.xlabel("degree k")
plt.ylabel("P(k)")
plt.title("Karate club degree distribution")

In [ ]:
# 核对（不用修改）
assert k_mean is not None and rho is not None, "k_mean、rho 还没有赋值"
assert abs(k_mean - 4.588) < 0.01, "平均度应约为 4.59 = 2×78/34；检查是否漏乘了 2"
assert abs(rho - 0.139) < 0.002, "密度应约为 0.139；分母是可能的最大边数 N(N-1)/2"
assert abs(rho - nx.density(G)) < 1e-9, "你的密度与 nx.density 不一致"
assert hub == 33 and k_max == 17, "度最大的是 33 号节点（度为 17）"
print("✓ 任务 1 通过")

## 任务 2：自己写广度优先搜索（BFS）

用队列实现 BFS，返回从源点到所有可达节点的距离字典。然后：

1. 求从 0 号节点出发的距离 `dist0`；
2. 对每个节点各做一次 BFS，得到所有节点对的距离，由此求直径 `diam`（最大距离）和平均路径长度 `L_avg`（所有无序节点对距离的平均）。

期望看到：与 `nx.single_source_shortest_path_length` 完全一致；直径为 5，平均路径长度约 2.41。

> **提示：** 在 TODO 处写 `if u not in dist: dist[u] = dist[v] + 1; queue.append(u)`。

In [ ]:
def bfs_distances(G, source):
    dist = {source: 0}
    queue = deque([source])
    while queue:
        v = queue.popleft()
        for u in G.neighbors(v):
            if u not in dist:
                dist[u] = dist[v] + 1
                queue.append(u)
    return dist


dist0 = bfs_distances(G, 0)
all_d = []
for s in G:
    ds = bfs_distances(G, s)
    all_d += [d for t, d in ds.items() if t > s]     # 每个无序对只计一次
diam = max(all_d)
L_avg = np.mean(all_d)
layers = {}
for node, d in dist0.items():
    layers.setdefault(d, []).append(node)
for d in sorted(layers):
    print("距离 %d: %s" % (d, sorted(layers[d])))
print("直径 = %d，平均路径长度 = %.3f" % (diam, L_avg))

In [ ]:
# 核对（不用修改）
assert len(dist0) == N, "dist0 只有 %d 个节点；检查 BFS 是否把未访问的邻居放入了队列" % len(dist0)
assert dist0 == dict(nx.single_source_shortest_path_length(G, 0)), "BFS 距离与 NetworkX 不一致；只有第一次遇到某节点时才记录距离"
assert diam == 5, "空手道网络的直径为 5"
assert abs(L_avg - 2.408) < 0.005, "平均路径长度应约为 2.408；检查每个节点对是否只算一次、是否排除了自己到自己的 0"
print("✓ 任务 2 通过")

## 任务 3：局部聚集系数、平均聚集系数与传递性

1. 手工数出 0 号节点的邻居之间有多少条边 `e_0`，算出 $C_0 = 2e_0/[k_0(k_0-1)]$，存为 `C_0`；
2. 用 `nx.average_clustering` 求平均聚集系数 `C_avg`；
3. **按定义**计算传递性 `T` $=3\times$三角形数$\,/\,\sum_i k_i(k_i-1)/2$，并与 `nx.transitivity` 核对。

期望看到：$C\approx0.57$，$T\approx0.26$，相差一倍以上。想想为什么低度节点会把平均聚集系数拉高。

> **提示：** `sum(1 for u, v in combinations(nbrs, 2) if G.has_edge(u, v))`；三角形数可用 `sum(nx.triangles(G).values()) / 3`。

In [ ]:
nbrs = list(G.neighbors(0))
k0 = len(nbrs)
e_0 = sum(1 for u, v in combinations(nbrs, 2) if G.has_edge(u, v))
C_0 = 2 * e_0 / (k0 * (k0 - 1))
C_avg = nx.average_clustering(G)
n_triangles = sum(nx.triangles(G).values()) / 3
n_triples = sum(d * (d - 1) / 2 for _, d in G.degree())
T = 3 * n_triangles / n_triples
print("节点 0: 度=%d, 邻居间边数=%d, C_0=%.3f（NetworkX %.3f）" % (k0, e_0, C_0, nx.clustering(G, 0)))
print("平均聚集系数 %.3f，传递性 %.3f（NetworkX %.3f）" % (C_avg, T, nx.transitivity(G)))

In [ ]:
# 核对（不用修改）
assert e_0 is not None and C_0 is not None and C_avg is not None and T is not None, "还有变量没有赋值"
assert e_0 == 18, "0 号节点的 16 个邻居之间有 18 条边；检查是否把每对邻居只数了一次"
assert abs(C_0 - nx.clustering(G, 0)) < 1e-9, "C_0 应为 2e/(k(k-1)) = 0.15"
assert abs(C_avg - 0.571) < 0.002, "平均聚集系数应约为 0.571"
assert abs(T - 0.256) < 0.002, "传递性应约为 0.256；三角形数要除以 3，连通三元组数为 Σ k(k-1)/2"
assert abs(T - nx.transitivity(G)) < 1e-9, "你算的传递性与 nx.transitivity 不一致"
print("✓ 任务 3 通过")

## 任务 4：中心性与 k 核

**(a) 手算核对。**星形图（1 个中心、4 个叶子）中，中心的（未归一化）介数应为 $\binom42=6$，叶子的接近中心性应为 $4/7\approx0.57$。用 NetworkX 求出 `bc_center`（`normalized=False`）和 `cc_leaf`。

**(b) 空手道网络。**
1. 求介数中心性最高的 3 个节点 `top_bc`（列表）；
2. 用 `nx.core_number` 求每个节点的核数，得到最大核数 `core_max` 与最内层核的节点数 `n_inner`。

期望看到：介数前三是 0、33、32；最大核数为 4，4 核包含 10 个节点（含 0 与 33）。

> **提示：** `sorted(bc, key=bc.get, reverse=True)[:3]`。

In [ ]:
# 准备（直接运行）
star = nx.star_graph(4)    # 中心为 0，叶子为 1–4

In [ ]:
bc_center = nx.betweenness_centrality(star, normalized=False)[0]
cc_leaf = nx.closeness_centrality(star)[1]
bc = nx.betweenness_centrality(G)
top_bc = sorted(bc, key=bc.get, reverse=True)[:3]
core = nx.core_number(G)
core_max = max(core.values())
n_inner = sum(1 for v in core.values() if v == core_max)
print("星形图：中心介数 %.0f，叶子接近中心性 %.3f" % (bc_center, cc_leaf))
print("介数前三:", [(n, round(bc[n], 3)) for n in top_bc])
print("最大核数 %d，最内层 %d 个节点: %s" % (core_max, n_inner, sorted(n for n, v in core.items() if v == core_max)))

In [ ]:
# 核对（不用修改）
assert bc_center is not None and cc_leaf is not None, "bc_center、cc_leaf 还没有赋值"
assert abs(bc_center - 6) < 1e-9, "星形中心的未归一化介数应为 C(4,2)=6；记得 normalized=False"
assert abs(cc_leaf - 4 / 7) < 1e-6, "叶子的接近中心性应为 4/(1+3×2) = 4/7"
assert top_bc is not None and set(top_bc) == {0, 32, 33}, "介数最高的 3 个节点应为 0、33、32；检查排序是否用了 reverse=True"
assert core_max == 4, "空手道网络的最大核数为 4"
assert n_inner == 10, "4 核应包含 10 个节点"
print("✓ 任务 4 通过")

## 任务 5：ER 网络中巨连通分支的涌现

对 $N=1000$ 的 ER 网络，让平均度 $\langle k\rangle$ 从 0.2 增加到 4，记录最大连通分支所占比例，存为列表 `S_sim`（与 `k_values` 一一对应）。再用不动点迭代求解 $S = 1 - e^{-\langle k\rangle S}$，得到理论值 `S_th`，画在同一张图上。

期望看到：$\langle k\rangle<1$ 时几乎没有巨分支，$\langle k\rangle=1$ 附近陡然上升；$\langle k\rangle=3$ 时理论值约 0.94。

> **提示：** `len(max(nx.connected_components(H), key=len)) / N_er`；不动点迭代从 S=0.5 出发反复执行 `S = 1 - np.exp(-kv * S)`。

In [ ]:
# 准备（直接运行）
N_er = 1000
k_values = np.linspace(0.2, 4, 20)

In [ ]:
S_sim = []
for i, kv in enumerate(k_values):
    H = nx.fast_gnp_random_graph(N_er, kv / (N_er - 1), seed=i)
    S_sim.append(len(max(nx.connected_components(H), key=len)) / N_er)


def S_theory(kv, iters=500):
    S = 0.5
    for _ in range(iters):
        S = 1 - np.exp(-kv * S)
    return S


S_th = np.array([S_theory(kv) for kv in k_values])
plt.figure(figsize=(5, 3))
plt.plot(k_values, S_sim, "o", label="simulation (N=1000)")
plt.plot(k_values, S_th, "-", label="theory")
plt.axvline(1, ls="--", c="gray")
plt.xlabel("average degree <k>")
plt.ylabel("fraction in giant component")
plt.legend()
print("<k>=3 附近: 模拟 %.3f, 理论 %.3f" % (S_sim[14], S_th[14]))

In [ ]:
# 核对（不用修改）
S_sim, S_th = np.asarray(S_sim, dtype=float), np.asarray(S_th, dtype=float)
assert S_sim.shape == k_values.shape, "S_sim 应有 %d 个值；检查循环里是否追加了比例" % len(k_values)
assert S_th.shape == k_values.shape, "S_th 应与 k_values 等长"
assert np.all(S_th[k_values < 0.95] < 1e-3), "<k> < 1 时理论巨分支比例应为 0"
assert abs(float(np.interp(3.0, k_values, S_th)) - 0.940) < 0.01, "<k>=3 时理论值应约为 0.94；检查迭代次数是否足够"
assert np.all(S_sim[k_values < 0.6] < 0.05), "<k> 很小时最大分支只是小碎片；检查是否除以了 N"
assert np.max(np.abs(S_sim - S_th)[k_values >= 1.5]) < 0.06, "<k> ≥ 1.5 时模拟值应与理论值接近"
print("✓ 任务 5 通过")

## 任务 6：小分析：真实网络与随机网络的比较

以《悲惨世界》人物共现网络（按无权图处理）为例，生成 20 个与它节点数、边数相同的 ER 网络（种子 0–19），在最大连通分支上计算平均聚集系数和平均路径长度。

把结果整理成一个字典 `summary`，包含以下键（都是正数）：

- `"C"`、`"L"`、`"k_max"`：真实网络的平均聚集系数、平均路径长度、最大度；
- `"C_rand"`、`"L_rand"`、`"k_max_rand"`：20 个 ER 网络的平均值。

再画一张图（例如两种网络度分布的对比），用一两句话回答：哪些性质 ER 模型能解释，哪些不能？
本任务是开放的，只检查结构和取值范围。

In [ ]:
# 准备（直接运行）
G_les = nx.Graph()
G_les.add_edges_from(nx.les_miserables_graph().edges())
n_les, m_les = G_les.number_of_nodes(), G_les.number_of_edges()

In [ ]:
def lcc(H):
    return H.subgraph(max(nx.connected_components(H), key=len)).copy()


rows = []
for s in range(20):
    R = nx.gnm_random_graph(n_les, m_les, seed=s)
    Rl = lcc(R)
    rows.append((nx.average_clustering(Rl), nx.average_shortest_path_length(Rl), max(d for _, d in R.degree())))
rows = np.array(rows)
summary = {"C": nx.average_clustering(G_les), "L": nx.average_shortest_path_length(G_les),
           "k_max": max(d for _, d in G_les.degree()),
           "C_rand": rows[:, 0].mean(), "L_rand": rows[:, 1].mean(), "k_max_rand": rows[:, 2].mean()}
for key, val in summary.items():
    print("%-10s %.3f" % (key, val))
deg_real = [d for _, d in G_les.degree()]
deg_rand = [d for _, d in nx.gnm_random_graph(n_les, m_les, seed=0).degree()]
plt.figure(figsize=(5, 3))
plt.hist([deg_real, deg_rand], bins=range(0, 38, 2), label=["Les Miserables", "ER"])
plt.xlabel("degree")
plt.ylabel("count")
plt.legend()
# 路径长度两者接近，ER 能解释“路径短”；聚集系数和最大度则远高于 ER，ER 无法解释。

In [ ]:
# 核对（不用修改）
assert isinstance(summary, dict), "summary 应是一个字典"
for key in ("C", "L", "k_max", "C_rand", "L_rand", "k_max_rand"):
    assert summary.get(key) is not None, "summary['%s'] 还没有填写" % key
    assert float(summary[key]) > 0, "summary['%s'] 应为正数" % key
assert 0 < summary["C"] <= 1 and 0 < summary["C_rand"] <= 1, "聚集系数应在 0–1 之间"
assert 1 < summary["L"] < n_les and 1 < summary["L_rand"] < n_les, "平均路径长度应大于 1、小于节点数；ER 网络记得在最大连通分支上计算"
assert summary["k_max"] < n_les and summary["k_max_rand"] < n_les, "最大度不可能超过 N-1"
print("✓ 任务 6 通过")